# CDR Data Preprocessing
Reads Call Detail Record files dropped into `data/cdr_incoming/` (the SFTP landing folder) and runs them through cleaning, encoding, and scaling before they reach the anomaly detection / forecasting / clustering models. Re-run this notebook top to bottom whenever new CDR files have landed.

In [2]:
from pathlib import Path  # object-oriented filesystem path handling

import numpy as np  # used later for np.log1p() when scaling duration
import pandas as pd  # the DataFrame/Series machinery this notebook is built on

# Path.cwd() is ML/preprocessing/ when the notebook is run from its own
# folder; .parent steps up to ML/, then down into ML/data.
DATA_DIR = Path.cwd().parent / "data"
INCOMING_DIR = DATA_DIR / "cdr_incoming"  # where new, unprocessed CDR files land
PROCESSED_DIR = DATA_DIR / "processed"  # where the cleaned, appended-to CSV lives
ARCHIVE_DIR = DATA_DIR / "archive"  # where source files go once processed

# How many interquartile ranges beyond Q1/Q3 counts as a duration outlier
# (see the Outliers section below).
OUTLIER_IQR_MULTIPLIER = 1.5
# Pearson correlation above this, between two numeric columns, means one
# of them is redundant (see the Redundant columns section below).
REDUNDANCY_CORR_THRESHOLD = 0.9

# Maps every raw header spelling seen across the different CDR exports
# (lowercased, since the lookup below always lowercases first) onto one
# canonical column name, so every downstream cell only has to know one
# set of names.
RAW_COLUMNS = {
    "date": "date",  # already canonical, maps to itself
    "orig. time": "orig_time",  # two spelling variants of the same column...
    "orig time": "orig_time",  # ...the originating call's time
    "orig.": "orig",  # two spellings of the originating-number column
    "orig": "orig",
    "dest.": "dest",  # two spellings of the destination-number column
    "dest": "dest",
    "call classification": "call_classification",  # the telco's own label
    "duration (sec)": "duration_sec",  # two spellings of the duration column
    "duration": "duration_sec",
    "charge": "charge",  # per-call cost, only present on billed exports
}
# The canonical column names every parsed file must end up with — the
# unique values of RAW_COLUMNS above.
REQUIRED_COLUMNS = set(RAW_COLUMNS.values())
# File extensions this notebook knows how to parse; anything else sitting
# in cdr_incoming/ (e.g. a stray .pdf) is ignored below.
SUPPORTED_EXTENSIONS = {".csv", ".txt", ".tsv"}

## Load CDR files
Every supported file currently sitting in `cdr_incoming/` gets read and concatenated. Files that don't match the expected CDR schema (e.g. a scanned PDF with no machine-readable table) are skipped with a warning rather than failing the whole run, since the folder accumulates files of mixed provenance over time.

In [3]:
def derive_cucm_fields(raw: pd.DataFrame) -> pd.DataFrame:
    """Map a real Cisco CallManager CDR export onto the columns normalize_columns expects.

    This export's own column names (dateTimeOrigination, callingPartyNumber,
    finalCalledPartyNumber, duration, ...) don't match RAW_COLUMNS, and it has
    no call-classification/charge columns at all since it's an internal
    cluster's own CDRs, not a billed telco feed.
    """
    # Build a lowercased-header -> original-header lookup so the checks
    # below work regardless of how this file capitalized its columns.
    cols = {c.strip().lower(): c for c in raw.columns}
    # Neither telltale CUCM column is present — this isn't a CUCM export,
    # return unchanged and let normalize_columns() handle it normally.
    if "datetimeorigination" not in cols or "callingpartynumber" not in cols:
        return raw

    # Copy before mutating, so the caller's DataFrame isn't affected.
    raw = raw.copy()
    # CUCM's dateTimeOrigination is a Unix epoch timestamp (seconds since
    # 1970) stored as text; coerce to numeric (errors="coerce" turns
    # anything unparseable into NaN) then to a real pandas datetime.
    origination = pd.to_datetime(pd.to_numeric(raw[cols["datetimeorigination"]], errors="coerce"), unit="s")
    # Split the single datetime back into separate date/time strings to
    # match RAW_COLUMNS's expected "date" + "orig_time" schema.
    raw["date"] = origination.dt.strftime("%Y-%m-%d")
    raw["orig_time"] = origination.dt.strftime("%H:%M:%S")
    # CUCM's callingPartyNumber is this schema's "orig" (who placed the call).
    raw["orig"] = raw[cols["callingpartynumber"]]
    # Prefer the final destination (after any forwarding); fall back to
    # the originally-dialled number if this export doesn't have that.
    dest_col = cols.get("finalcalledpartynumber") or cols.get("originalcalledpartynumber")
    raw["dest"] = raw[dest_col]
    if "duration" in cols:
        # Copy into the canonical name, then drop the original-cased
        # column so it isn't left behind as a duplicate.
        raw["duration_sec"] = raw[cols["duration"]]
        raw = raw.drop(columns=[cols["duration"]])
    raw["charge"] = 0  # CUCM CDRs carry no billing charge; internal calls bill at 0
    raw["call_classification"] = "Internal"  # this export is an internal cluster's own CDRs
    return raw


def normalize_columns(raw: pd.DataFrame) -> pd.DataFrame:
    # Synthesize CUCM-shaped columns first if this is a CUCM export;
    # a no-op for every other export (see derive_cucm_fields's early return).
    raw = derive_cucm_fields(raw)
    # Strip + lowercase every column name and look it up in RAW_COLUMNS,
    # falling back to the lowercased-stripped name itself if it's not a
    # known alias (so an already-canonical column passes through as-is).
    raw = raw.rename(columns={c: RAW_COLUMNS.get(c.strip().lower(), c.strip().lower()) for c in raw.columns})
    # Set difference: any canonical column still missing after renaming.
    missing = REQUIRED_COLUMNS - set(raw.columns)
    if missing:
        # sorted() purely for a deterministic, readable error message.
        raise ValueError(f"missing expected columns: {sorted(missing)}")
    # Subset (and implicitly reorder) to just the columns this pipeline
    # relies on downstream, dropping anything extra the file had.
    return raw[list(REQUIRED_COLUMNS)]


def read_one(path: Path) -> pd.DataFrame:
    # Tab-separated for .tsv, comma-separated for everything else.
    sep = "\t" if path.suffix.lower() == ".tsv" else ","
    # dtype=str forces every column to be read as text rather than
    # guessed — critical so a phone-number-like column never gets
    # silently turned into a number (losing a leading "0", etc.).
    file_df = pd.read_csv(path, sep=sep, dtype=str)
    file_df = normalize_columns(file_df)  # rename/derive to the canonical schema
    file_df["source_file"] = path.name  # remember which file each row came from
    return file_df


# List every regular file in cdr_incoming/ with a supported extension,
# sorted for a deterministic processing order.
files = sorted(p for p in INCOMING_DIR.iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED_EXTENSIONS)
frames = []  # successfully parsed DataFrames, one per file
for path in files:
    try:
        # Parse + normalize this one file and add it to the pile.
        frames.append(read_one(path))
    except (ValueError, pd.errors.ParserError) as exc:
        # A malformed or unrecognised-schema file — skip it rather than
        # aborting the whole batch, and print why.
        print(f"skipping {path.name}: {exc}")

# Stack every parsed file's rows into one DataFrame; if nothing parsed,
# fall back to an empty DataFrame with the right columns so the rest of
# the notebook can still safely run against it.
df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=list(REQUIRED_COLUMNS) + ["source_file"])
print(f"loaded {len(df)} records from {len(frames)} file(s)")
df.head(10)  # eyeball the first 10 parsed rows

loaded 0 records from 0 file(s)
Empty DataFrame
Columns: [dest, duration_sec, orig, date, orig_time, call_classification, charge, source_file]
Index: []


In [4]:
# Column dtypes, non-null counts, and memory usage — a quick sanity
# check that every expected column is present and roughly the right type.
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 0 entries
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   dest                 0 non-null      object
 1   duration_sec         0 non-null      object
 2   orig                 0 non-null      object
 3   date                 0 non-null      object
 4   orig_time            0 non-null      object
 5   call_classification  0 non-null      object
 6   charge               0 non-null      object
 7   source_file          0 non-null      object
dtypes: object(8)
memory usage: 124.0+ bytes


## Missing values
QoS is never populated by the telco (always `NA`) — drop it rather than impute.

In [5]:
# errors="ignore" means this doesn't raise if "qos" isn't present at
# all in this particular batch of files.
df = df.drop(columns=["qos"], errors="ignore")
df.columns.tolist()  # quick look at the remaining column names
df.head()  # preview the first 5 rows after dropping qos

,dest,duration_sec,orig,date,orig_time,call_classification,charge,source_file


## Noise
Parse Date + Orig. Time into a single timestamp, coerce Duration/Charge to numeric, and drop records that are physically invalid: negative durations or charges, or timestamps that couldn't be parsed at all.

In [4]:
# Concatenate the separate date + time text columns into one string,
# then parse into a real pandas datetime; errors="coerce" turns any row
# whose date/time text doesn't parse into NaT instead of raising.
df["call_timestamp"] = pd.to_datetime(df["date"].astype(str) + " " + df["orig_time"].astype(str), errors="coerce")
# Coerce these text columns to numeric; anything non-numeric becomes NaN.
df["duration_sec"] = pd.to_numeric(df["duration_sec"], errors="coerce")
df["charge"] = pd.to_numeric(df["charge"], errors="coerce")

before = len(df)  # row count before filtering, just to log the drop count
# Drop rows whose timestamp never parsed (NaT) — physically meaningless.
df = df.dropna(subset=["call_timestamp"])
# Keep only non-negative duration and charge; fillna(0) means a
# *missing* value doesn't get dropped here (0 >= 0), only a genuinely
# negative one does.
df = df[(df["duration_sec"].fillna(0) >= 0) & (df["charge"].fillna(0) >= 0)]
print(f"dropped {before - len(df)} noisy record(s), {len(df)} remain")
df.head()

dropped 0 noisy record(s), 0 remain


,date,orig,dest,orig_time,duration_sec,call_classification,charge,source_file,call_timestamp


## Outliers
Flag statistical outliers in call duration via IQR fencing — **not removed**, since these flagged rows are exactly what the Isolation Forest anomaly detection model needs to see downstream.

In [5]:
# Q1 (25th percentile) and Q3 (75th percentile) of call duration.
q1, q3 = df["duration_sec"].quantile([0.25, 0.75])
# Interquartile range: the spread of the middle 50% of the data.
iqr = q3 - q1
# Classic Tukey fence: more than OUTLIER_IQR_MULTIPLIER IQRs below Q1 or
# above Q3 counts as an outlier.
lower, upper = q1 - OUTLIER_IQR_MULTIPLIER * iqr, q3 + OUTLIER_IQR_MULTIPLIER * iqr
# .between() is True *inside* [lower, upper]; the leading ~ inverts it,
# so is_outlier is True for anything *outside* the fence.
df["is_outlier"] = ~df["duration_sec"].between(lower, upper)
print(f"{df['is_outlier'].sum()} of {len(df)} records flagged as duration outliers")
df[df["is_outlier"]].head()  # peek at a few of the flagged rows

0 of 0 records flagged as duration outliers


,date,orig,dest,orig_time,duration_sec,call_classification,charge,source_file,call_timestamp,is_outlier


## Redundant columns
Compute the Pearson correlation matrix across numeric columns and drop one column from any pair that's redundant (correlated above the threshold).

In [6]:
# Only numeric columns can be correlated against each other.
numeric = df.select_dtypes(include="number")
# Full pairwise Pearson correlation matrix, absolute value (a strong
# negative correlation is just as redundant as a strong positive one).
corr = numeric.corr(method="pearson").abs()
corr  # display the matrix

,duration_sec,charge
duration_sec,NaN,NaN
charge,NaN,NaN


In [7]:
to_drop = set()  # column names queued for removal
# Iterate every unique pair of columns (col_b always comes after col_a,
# so each pair is only considered once).
for i, col_a in enumerate(corr.columns):
    for col_b in corr.columns[i + 1:]:
        # Skip a pair if either column is already marked for removal
        # from an earlier, related pair.
        if col_a in to_drop or col_b in to_drop:
            continue
        if corr.loc[col_a, col_b] > REDUNDANCY_CORR_THRESHOLD:
            to_drop.add(col_b)  # arbitrarily drop the second column of the pair

print(f"dropping redundant column(s): {sorted(to_drop) or 'none'}")
df = df.drop(columns=list(to_drop))

dropping redundant column(s): none


## Encoding
One-hot encode Call Classification (Internal / Others / ...).

In [8]:
# One-hot encode: replaces the single call_classification column with
# one boolean column per distinct value actually present in this batch
# (call_class_Internal, call_class_External, ...), each True/False for
# whether that row was that classification.
df = pd.get_dummies(df, columns=["call_classification"], prefix="call_class")
df.head()

,date,orig,dest,orig_time,duration_sec,charge,source_file,call_timestamp,is_outlier


## Feature scaling
Duration has a long right tail (a handful of very long calls). Log-transform it toward a bell curve, then standardize to zero mean / unit variance.

In [9]:
# log1p(x) = log(1 + x) — avoids log(0) being undefined for zero-second
# calls, and compresses duration's long right tail (a handful of very
# long calls) toward a more bell-shaped distribution. fillna(0) treats
# a still-missing duration as zero.
log_duration = np.log1p(df["duration_sec"].fillna(0))
# Standard deviation of the log-transformed values, for z-score scaling.
std = log_duration.std()
# Standard z-score: (x - mean) / std, giving zero mean and unit variance
# — guarded against std being 0 (every call the same duration), in
# which case every scaled value is just set to 0.0 instead of NaN/inf.
df["duration_sec_scaled"] = (log_duration - log_duration.mean()) / std if std else 0.0
# Compare the raw vs. scaled distributions (mean, std, min/max, etc.).
df[["duration_sec", "duration_sec_scaled"]].describe()

,duration_sec,duration_sec_scaled
count,0.0,0.0
mean,NaN,NaN
std,NaN,NaN
min,NaN,NaN
25%,NaN,NaN
50%,NaN,NaN
75%,NaN,NaN
max,NaN,NaN


## Feature selection
Drop Charge — it isn't predictive for usage-pattern analysis (nearly all internal calls bill at 0) and was only kept around for the redundancy check above. Call records are **not** deduplicated: the same two extensions can legitimately call each other multiple times a day, so identical rows are real, distinct events, not duplicates to collapse.

In [10]:
# See the markdown above for why charge is dropped here.
df = df.drop(columns=["charge"], errors="ignore")
df.head()

,date,orig,dest,orig_time,duration_sec,source_file,call_timestamp,is_outlier,duration_sec_scaled


## Save
Append the cleaned batch to the processed dataset, then move the ingested files out of `cdr_incoming/` so the next run only picks up newly arrived files.

In [13]:
# Make sure the output directory exists before writing to it.
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
out_path = PROCESSED_DIR / "cdr_preprocessed.csv"
# Only write a header row the very first time this file is created —
# every later run appends without repeating the header.
header = not out_path.exists()
# mode="a" appends rather than overwrites; index=False so pandas
# doesn't add its own row-number column.
df.to_csv(out_path, mode="a", header=header, index=False)

# Make sure the archive directory exists before moving files into it.
ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)
for path in files:
    # Move (not copy) each processed source file into the archive, so
    # cdr_incoming/ only ever holds genuinely new files for next time.
    path.rename(ARCHIVE_DIR / path.name)

print(f"wrote {len(df)} records -> {out_path}")
df.head(10)  # preview the tail end of what just got written

wrote 0 records -> /home/mugo/Projects/TeleSpy/ML/data/processed/cdr_preprocessed.csv
Empty DataFrame
Columns: [date, orig, dest, orig_time, duration_sec, source_file, call_timestamp, is_outlier, duration_sec_scaled]
Index: []
